# 03 · Error analysis — why the wrong answers were wrong

**The thesis: an accuracy number says how often; only a human says why.** This
is the diagnostic half of the paper, and it is yours.

No API key needed.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `pull the failures` | every row the scorer marked wrong | `df[df.correct == 0]` |
| `the five categories` | how each failure is classified | `label_wrong`, `wrong_cell`, ... |
| `tool_calls` | did the agent reach for a tool on this question? | 0, 1, 3 ... |

## Step 1 — load the results and isolate the failures

In [1]:
import glob, os
import pandas as pd
paths = sorted(glob.glob("results-input/*.csv")) + \
        sorted(glob.glob("../rashan-table-agent/results/*.csv"))
df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
wrong = df[df["correct"] == 0]
print(f"{len(wrong)} wrong of {len(df)} rows")
print(wrong.groupby("config").size().to_string())

80 wrong of 340 rows
config
agent                23
baseline             22
chatgpt_reference    14
cot                  21


## Step 2 — the five categories

For each wrong answer, decide which one it is. Only a person can do this.

```
 label_wrong    the AI was right; the benchmark's gold is wrong
 wrong_cell     it used the wrong row or column
 math_error     right cells, wrong arithmetic
 format_only    same answer, written differently - the meaning matches
 unanswerable   the table genuinely does not contain the answer
```

`label_wrong` and `unanswerable` are the important ones: they are *not* model
failures, and counting them as such makes the system look worse than it is.

## Step 3 — look at real failures before you decide anything

Read the question, the gold, and what the model said. Do not skim.

In [2]:
cols = [c for c in ("config","question","gold","pred","reasoning_type","tool_calls")
        if c in wrong.columns]
for _, r in wrong[cols].head(8).iterrows():
    print("-"*78)
    print("config :", r.get("config"))
    print("Q      :", str(r["question"])[:96])
    print("gold   :", r["gold"], "   model:", repr(r["pred"]))
    if "tool_calls" in r: print("tools  :", r["tool_calls"])

------------------------------------------------------------------------------
config : chatgpt_reference
Q      : What is the total length of the sequence formed by concatenating the forward and reverse primers
gold   : 48 nucleotides    model: '60 nucleotides'
tools  : nan
------------------------------------------------------------------------------
config : chatgpt_reference
Q      : What is the difference in length between the forward and reverse primers for Exon 2?
gold   : 3 nucleotides    model: '20 nucleotides'
tools  : nan
------------------------------------------------------------------------------
config : chatgpt_reference
Q      : What is the total length of the sequence formed by concatenating the forward and reverse primers
gold   : 46 nucleotides    model: '45 nucleotides'
tools  : nan
------------------------------------------------------------------------------
config : chatgpt_reference
Q      : What is the total length of the sequence formed by concatenating the f

## Step 4 — the worked example

> gold `48 nucleotides`; the table gives a forward sequence of 24 characters and
> a reverse of 22; 24 + 22 = 46; the model answered 46.

The model is right. The gold cannot be obtained from the table we were given.
That is `label_wrong`, and if it is common it changes how the whole paper
reports accuracy.

## Step 5 — did using a tool help or hurt?

If `tool_calls` is present, compare accuracy when the agent used a tool against
when it did not. Careful with the conclusion: the agent reaches for a tool on
*harder* questions, so a lower score there may be the questions, not the tool.

In [3]:
a = df[df["config"].astype(str).str.startswith("agent")]
if len(a) and "tool_calls" in a.columns:
    used = a[a["tool_calls"] > 0]; non = a[a["tool_calls"] == 0]
    print(f"tool used     : {used['correct'].mean():.1%}  (n={len(used)})" if len(used) else "no tool-using rows")
    print(f"no tool used  : {non['correct'].mean():.1%}  (n={len(non)})" if len(non) else "")
    print()
    print("To separate 'the tool hurt' from 'those questions are hard',")
    print("compare the BASELINE on the same questions.")
else:
    print("no agent rows with tool_calls yet")

tool used     : 28.6%  (n=14)
no tool used  : 84.9%  (n=86)

To separate 'the tool hurt' from 'those questions are hard',
compare the BASELINE on the same questions.


## Step 6 — record it

Fill `audit/audit_sheet.csv` (one row per item) and summarise the counts in
`results/NOTES.md`. The summary table the paper needs is simply:

```
 category        count    share of failures
 label_wrong       ..            ..
 wrong_cell        ..            ..
 math_error        ..            ..
 format_only       ..            ..
 unanswerable      ..            ..
```

If many failures are `label_wrong` or `unanswerable`, say so loudly — it is a
finding about the benchmark, not a weakness in your analysis.

## What to carry out of this notebook

```
 · accuracy says how often; categories say why
 · label_wrong and unanswerable are NOT model failures
 · tool use is confounded with question difficulty - control for it
 · ambiguous cases are findings; write them down rather than guessing
```